# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (not as a dict)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List all record sets by their @id and name from metadata
print("Available RecordSets:")
record_sets = list(meta.record_sets)
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name}")

# For each record set, list its fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    if rs.fields:
        print("  Fields:")
        for fld in rs.fields:
            print(f"    @id: {fld.id}, name: {fld.name}, data_type: {fld.data_type}, column: {fld.column.id if fld.column else None}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare a list of record set @id values to extract data from
record_set_ids = [rs.id for rs in record_sets]
dataframes = dict()

# Load each record set as a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, pick the first record set with data
main_record_set_id = None
for record_set_id, df in dataframes.items():
    if df.shape[0] > 0:
        main_record_set_id = record_set_id
        break
if main_record_set_id is not None:
    print(f"Main record set selected: {main_record_set_id}")
    print("Columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No populated record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA for a numeric field from the main record set
df = dataframes[main_record_set_id].copy() if main_record_set_id else pd.DataFrame()
if not df.empty:
    # Try to automatically detect a numeric field (float or int)
    numeric_fields = df.select_dtypes(include=["float", "int"]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Take the first numeric field
        print(f"Using numeric field for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}: {filtered_df.shape[0]}")
        display(filtered_df[[numeric_field]].head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical column
        group_field = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data in main DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping was possible
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df.columns[0], y=grouped_df.columns[1], data=grouped_df)
        plt.title(f"Mean {numeric_field} by {grouped_df.columns[0]}")
        plt.xlabel(grouped_df.columns[0])
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("Visualization skipped: suitable numeric field or data not found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.